# Evaluate the exported ALS baseline

This notebook evaluates an already-exported top-10 ALS baseline against the held-out MovieLens ratings. It does not fit ALS models or write recommendation exports. The training/test split is rebuilt with the same deterministic code used by `popularity-profiles.ipynb` so relevance labels match the baseline evaluation.


## Configure the baseline run

Set `baselineRunId` to an existing folder under `outputs/als-baseline/`. The optional historical regression check applies only to the baseline run and expected metrics listed here.


In [ ]:
val baselineRunId = "2026-11-15T22-59-34.326855Z"
val checkHistoricalBaselineRegression = false
val historicalRegressionBaselineRunId = "2026-11-15T22-59-34.326855Z"
val historicalPrecision = 0.0567
val historicalRecall = 0.0391
val historicalNdcg = 0.0513
val historicalMetricTolerance = 5e-4


## Start Spark and verify the runtime

Use the project runtime (Spark 3.5.9, Scala 2.12.18, Java 17) and a local two-thread Spark session. The synthetic count confirms Spark can execute before reading project data.


In [ ]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`
import $ivy.`org.apache.spark::spark-mllib:3.5.9`

import org.apache.spark.sql.SparkSession


In [ ]:
val spark = SparkSession.builder()
  .appName("CS 7265 - Accuracy Evaluation")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val scalaVersion = scala.util.Properties.versionNumberString
val javaVersion = System.getProperty("java.version")
val environmentCheck = spark.range(3).count()

assert(spark.version == "3.5.9")
assert(scalaVersion == "2.12.18")
assert(System.getProperty("java.specification.version") == "17")
assert(environmentCheck == 3L)

println(s"Spark: ${spark.version}")
println(s"Scala: $scalaVersion")
println(s"Java: $javaVersion")
println(s"PASS: environment check on $environmentCheck synthetic rows")


## Load MovieLens ratings

Read `data/ml-1m/ratings.dat` from the repository root or `notebooks/` working directory. The parser retains user, movie, rating, and timestamp fields; no model is fit in this notebook.


In [ ]:
case class Rating(userId: Int, movieId: Int, rating: Float, timestamp: Long)

class RatingParser extends (String => Rating) with Serializable {
  def apply(line: String): Rating = {
    val fields = line.split("::")
    require(fields.length == 4, "Expected four fields per MovieLens rating")

    Rating(
      fields(0).toInt,
      fields(1).toInt,
      fields(2).toFloat,
      fields(3).toLong
    )
  }
}

In [ ]:
val ratingsPath = Seq(
  "data/ml-1m/ratings.dat",
  "../data/ml-1m/ratings.dat"
).find(path => new java.io.File(path).isFile).getOrElse {
  throw new IllegalArgumentException("Place MovieLens 1M ratings.dat in data/ml-1m/ and run from the repository root or notebooks folder.")
}
val repositoryRoot = new java.io.File(ratingsPath).getCanonicalFile
  .getParentFile.getParentFile.getParentFile
val ratingsRDD = spark.sparkContext.textFile(ratingsPath).map(new RatingParser)
import spark.implicits._
val ratingsDF = ratingsRDD.toDF().cache()
println(s"PASS: loaded ${ratingsDF.count()} ratings from $ratingsPath")


## Rebuild the baseline train/test split

This is the deterministic split code from `popularity-profiles.ipynb`: per user, order ratings by `xxhash64(splitSeed, userId, movieId)` then `movieId`, assigning `floor(0.8 * userRatingCount)` rows to training. Verify the persisted baseline's expected 797,758 training and 202,451 test ratings and that no user/movie pair occurs in both.


In [ ]:
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

// Score the baseline against the holdout it was trained for: read the split parameters from the
// selected baseline run's split-config.txt instead of assuming seed 42.
val baselineRunDir = new java.io.File(repositoryRoot, s"outputs/als-baseline/$baselineRunId")
require(baselineRunDir.isDirectory, s"Baseline run not found: ${baselineRunDir.getPath}")
val baselineSplitConfigFile = new java.io.File(baselineRunDir, "split-config.txt")
val baselineSplitConfig: Map[String, String] =
  if (baselineSplitConfigFile.isFile) {
    new String(java.nio.file.Files.readAllBytes(baselineSplitConfigFile.toPath), "UTF-8")
      .split("\r?\n").filter(_.contains("=")).map { line =>
        val parts = line.split("=", 2)
        parts(0).trim -> parts(1).trim
      }.toMap
  } else {
    // Only the historical seed-42 baseline predates split-config.txt; its split is known.
    require(baselineRunId == historicalRegressionBaselineRunId,
      s"Baseline run $baselineRunId has no split-config.txt, so its split cannot be verified. Rerun the baseline notebook.")
    Map("splitSeed" -> "42", "trainingFraction" -> "0.8",
      "trainingRatings" -> "797758", "testRatings" -> "202451")
  }
def splitConfigValue(key: String): String = baselineSplitConfig.getOrElse(key,
  throw new IllegalArgumentException(s"split-config.txt for baseline $baselineRunId does not record $key."))
val splitSeed = splitConfigValue("splitSeed").toLong
val trainingFraction = splitConfigValue("trainingFraction").toDouble
val expectedTrainingRatingCount = splitConfigValue("trainingRatings").toLong
val expectedTestRatingCount = splitConfigValue("testRatings").toLong
require(Set(42L, 43L, 44L).contains(splitSeed), s"Unsupported split seed in baseline metadata: $splitSeed")
require(trainingFraction > 0.0 && trainingFraction < 1.0, s"Invalid training fraction: $trainingFraction")
println(s"PASS: baseline $baselineRunId uses splitSeed=$splitSeed, trainingFraction=$trainingFraction")

val perUser = Window.partitionBy("userId")
val ratingsWithCounts = ratingsDF.withColumn(
  "userRatingCount",
  count("*").over(perUser)
)

val ratingsWithKeys = ratingsWithCounts.withColumn(
  "shuffleKey",
  xxhash64(lit(splitSeed), col("userId"), col("movieId"))
)

val shuffledPerUser = Window
  .partitionBy("userId")
  .orderBy(col("shuffleKey"), col("movieId"))

val rankedRatings = ratingsWithKeys.withColumn(
  "ratingPosition",
  row_number().over(shuffledPerUser)
)

val ratingsWithCutoff = rankedRatings.withColumn(
  "trainingCount",
  floor(col("userRatingCount") * trainingFraction)
)

val training = ratingsWithCutoff.filter(
  col("ratingPosition") <= col("trainingCount")
)
val test = ratingsWithCutoff.filter(
  col("ratingPosition") > col("trainingCount")
)

val trainingRatings = training.select("userId", "movieId", "rating", "timestamp")
val testRatings = test.select("userId", "movieId", "rating", "timestamp")

In [ ]:
val trainingRatingCount = trainingRatings.cache().count()
val testRatingCount = testRatings.cache().count()
val trainingUserCount = trainingRatings.select("userId").distinct().count()
val testUserCount = testRatings.select("userId").distinct().count()
val splitOverlap = trainingRatings.select("userId", "movieId")
  .join(testRatings.select("userId", "movieId"), Seq("userId", "movieId"), "inner")
  .limit(1).count()

// The rebuilt split must reproduce the counts the baseline run recorded for its split.
assert(trainingRatingCount == expectedTrainingRatingCount,
  s"Expected $expectedTrainingRatingCount training ratings for seed $splitSeed, found $trainingRatingCount")
assert(testRatingCount == expectedTestRatingCount,
  s"Expected $expectedTestRatingCount test ratings for seed $splitSeed, found $testRatingCount")
assert(trainingUserCount == 6040L && testUserCount == 6040L,
  s"Expected 6040 users in both splits; training=$trainingUserCount, test=$testUserCount")
assert(splitOverlap == 0L, "Training and test splits overlap on user/movie pairs.")
println(s"PASS: split counts training=$trainingRatingCount, test=$testRatingCount; users=$trainingUserCount/$testUserCount; overlap=$splitOverlap")


## Load the persisted top-10 baseline

Read `top-10-baseline` from the selected run folder. Validate its schema and one complete, uniquely ranked list of ten movies per user. This notebook does not train a model or export/overwrite recommendations.


In [ ]:
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types.IntegerType

val baselineExportDir = new java.io.File(repositoryRoot,
  s"outputs/als-baseline/$baselineRunId/top-10-baseline")
require(baselineExportDir.isDirectory,
  s"Baseline top-10 export not found: ${baselineExportDir.getPath}")
val baselineRecommendations = spark.read.parquet(baselineExportDir.getPath).cache()

Seq("userId", "movieId", "recommendationRank").foreach { name =>
  require(baselineRecommendations.schema(name).dataType == IntegerType,
    s"$name must be Int in the baseline export.")
}
val recommendationCount = baselineRecommendations.count()
val recommendationUsers = baselineRecommendations.select("userId").distinct().count()
val invalidLists = baselineRecommendations.groupBy("userId")
  .agg(count("*").as("items"), min("recommendationRank").as("minRank"),
    max("recommendationRank").as("maxRank"), countDistinct("recommendationRank").as("ranks"))
  .filter(col("items") =!= 10L || col("minRank") =!= 1 ||
    col("maxRank") =!= 10 || col("ranks") =!= 10L)
  .count()
assert(recommendationCount == 60400L,
  s"Expected 60400 baseline recommendations, found $recommendationCount")
assert(recommendationUsers == 6040L,
  s"Expected 6040 baseline recommendation users, found $recommendationUsers")
assert(invalidLists == 0L, s"Found $invalidLists users without a complete ranked top-10 list.")
println(s"PASS: loaded $recommendationCount recommendations for $recommendationUsers users from ${baselineExportDir.getPath}")


## Choose which users to evaluate

A test movie counts as liked if the user gave it 4 or 5 stars. I only score users who have at least one liked movie in the test set, because recall has no meaning for a user with no liked movies. I print how many users are scored and how many are skipped, so the choice is clear and nothing is hidden. I also check that every scored user really has recommendations.

In [ ]:
// A liked movie = rated 4 or higher in the test set.
val relevant = testRatings.filter(col("rating") >= 4.0).select("userId", "movieId")

// Eligibility is defined from the TEST set first: users with at least one liked movie.
val eligibleUsers = relevant.select("userId").distinct()
val recUsers      = baselineRecommendations.select("userId").distinct()

// Independent check: every eligible user must have a recommendation (checked before joining).
val uncovered = eligibleUsers.join(recUsers, Seq("userId"), "left_anti").count()
assert(uncovered == 0L, "Every eligible user must have recommendations.")

// Users we evaluate = eligible users, now confirmed to all have recommendations.
val evaluatedUsers = eligibleUsers.join(recUsers, Seq("userId"), "inner").cache()

val totalWithRecs = recUsers.count()
val nEvaluated    = evaluatedUsers.count()

// Report who is scored and who is skipped.
println(s"Users with recommendations: $totalWithRecs")
println(s"Users evaluated:            $nEvaluated")
println(s"Users skipped:              ${totalWithRecs - nEvaluated}")
println("Check passed.")

## Accuracy scores

For each user I line up their 10 recommended movies against the movies they liked in the test set, then take the average over all users. Precision@10 is how many of the 10 recommended movies were liked. Recall@10 is how many of the user's liked movies showed up in the top 10. NDCG@10 also rewards putting liked movies near the top of the list. I use Spark's RankingMetrics to work these out.

In [ ]:
import org.apache.spark.mllib.evaluation.RankingMetrics

// Each user's 10 movies in order, best first.
val predicted = baselineRecommendations.groupBy("userId")
  .agg(sort_array(collect_list(struct(col("recommendationRank").as("r"),
                                      col("movieId").as("m")))).as("s"))
  .withColumn("pred", expr("transform(s, x -> x.m)"))
  .select("userId", "pred")

// Each user's set of liked movies.
val actual = relevant.groupBy("userId").agg(collect_set("movieId").as("act"))

// Keep only the scored users, then hand the lists to RankingMetrics.
val joined  = predicted.join(actual, Seq("userId")).select("pred", "act")
val rankRdd = joined.rdd.map { r => (r.getSeq[Int](0).toArray, r.getSeq[Int](1).toArray) }
val metrics = new RankingMetrics(rankRdd)

println(f"Users evaluated: ${joined.count()}")
println(f"Precision@10:    ${metrics.precisionAt(10)}%.4f")
println(f"Recall@10:       ${metrics.recallAt(10)}%.4f")
println(f"NDCG@10:         ${metrics.ndcgAt(10)}%.4f")

In [ ]:
if (checkHistoricalBaselineRegression) {
  require(baselineRunId == historicalRegressionBaselineRunId,
    s"Historical accuracy applies only to baseline $historicalRegressionBaselineRunId, not $baselineRunId.")
  val precisionAt10 = metrics.precisionAt(10)
  val recallAt10 = metrics.recallAt(10)
  val ndcgAt10 = metrics.ndcgAt(10)
  val checks = Seq(
    "P@10" -> (precisionAt10, historicalPrecision),
    "R@10" -> (recallAt10, historicalRecall),
    "NDCG@10" -> (ndcgAt10, historicalNdcg))
  checks.foreach { case (name, (actual, expected)) =>
    val passed = math.abs(actual - expected) <= historicalMetricTolerance
    println(s"${if (passed) "PASS" else "FAIL"}: $name actual=$actual expected=$expected tolerance=$historicalMetricTolerance")
    assert(passed, s"$name differs from the historical baseline regression value.")
  }
} else {
  println("SKIP: historical baseline regression assertions are disabled.")
}


## Validation tuning is separate

The 16-configuration ALS experiment is in [`als-tuning.ipynb`](als-tuning.ipynb). It compares validation RMSE, Precision@10, and NDCG@10 using a split of training data only; this evaluation notebook only loads the chosen, persisted baseline and scores it on the held-out test split.


## Stop Spark

Stop the local Spark session when the accuracy evaluation is complete.


In [ ]:
spark.stop()
println("PASS: Spark session stopped")
